# Nepali Sentiment Analysis

**Project:** End-to-end machine learning pipeline for Nepali-language sentiment classification.

**Sections:**
1. Problem Definition
2. Data Loading, Cleaning & EDA
3. Text Preprocessing (normalization, stopwords, TF-IDF)
4. Train/Test Split
5. Model Training (Naive Bayes, Logistic Regression, Linear SVM)
6. Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
7. Results Summary & Model Saving

## 1. Problem Definition

- **Dataset:** Nepali text sentiment dataset stored locally in `data/raw/nepali_sentiment_raw.csv`.
- **Task:** Binary or multi-class text classification, depending on the labels present in the dataset.
- **Target variable:** `label`, normalized from common sentiment column names such as `label`, `sentiment`, `class`, or `target`.
- **Objective:** Predict the sentiment of a Nepali text sample using classical machine learning models and TF-IDF features.

In [1]:
from pathlib import Path
import os
import warnings
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
MPL_CONFIG_DIR = PROJECT_ROOT / '.matplotlib'
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'nepali_sentiment_raw.csv'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'nepali_sentiment_clean.csv'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'

for directory in [FIGURES_DIR, RESULTS_DIR, MODELS_DIR, PROCESSED_DATA_PATH.parent]:
    directory.mkdir(parents=True, exist_ok=True)

## 2. Data Loading, Cleaning & EDA

In [2]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f'Raw dataset not found at {RAW_DATA_PATH}. Download the CSV and place it in data/raw/.'
    )

df = pd.read_csv(RAW_DATA_PATH)
df.head()

,text,label
0,यो फिल्म धेरै राम्रो छ,positive
1,मलाई यो सेवा मन पर्यो,positive
2,खाना निकै मिठो थियो,positive
3,अनुभव उत्कृष्ट भयो,positive
4,यो उत्पादन उपयोगी छ,positive


In [3]:
# Basic inspection
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.duplicated().sum())

(60, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    60 non-null     object
 1   label   60 non-null     object
dtypes: object(2)
memory usage: 1.1+ KB
None
text     0
label    0
dtype: int64
0


In [4]:
def find_column(columns, candidates):
    normalized = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    return None

text_col = find_column(df.columns, ['text', 'sentence', 'review', 'comment', 'content', 'tweet', 'message'])
label_col = find_column(df.columns, ['label', 'sentiment', 'class', 'target', 'category'])

if text_col is None or label_col is None:
    raise ValueError(
        'Could not identify text and label columns automatically. '
        f'Available columns: {list(df.columns)}'
    )

df = df.rename(columns={text_col: 'text', label_col: 'label'})
df = df[['text', 'label']].copy()
df['label'] = df['label'].astype(str).str.strip().str.lower()
print(f'Using text column: {text_col}')
print(f'Using label column: {label_col}')
df.head()

Using text column: text
Using label column: label


,text,label
0,यो फिल्म धेरै राम्रो छ,positive
1,मलाई यो सेवा मन पर्यो,positive
2,खाना निकै मिठो थियो,positive
3,अनुभव उत्कृष्ट भयो,positive
4,यो उत्पादन उपयोगी छ,positive


In [5]:
# Drop duplicates and missing values
df = df.drop_duplicates()
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].astype(str).str.strip().ne('')]
df.shape

(60, 2)

In [6]:
# Class balance
ax = df['label'].value_counts().plot(kind='bar', title='Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sentiment_distribution.png', dpi=150)
plt.show()

In [7]:
# Text length distribution
df['text_length'] = df['text'].astype(str).apply(len)
df['text_length'].hist(bins=40)
plt.title('Text Length Distribution')
plt.xlabel('Character count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'text_length_distribution.png', dpi=150)
plt.show()

## 3. Text Preprocessing

Steps: Unicode normalization, removing punctuation/numerals, Nepali stopword removal, TF-IDF vectorization.

In [8]:
import unicodedata

def clean_text(text):
    text = str(text)
    text = unicodedata.normalize('NFC', text)  # normalize Devanagari encoding variants
    text = re.sub(r'[^\u0900-\u097F\s]', ' ', text)  # keep only Devanagari + whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].ne('')].copy()
df[['text', 'clean_text']].head()

,text,clean_text
0,यो फिल्म धेरै राम्रो छ,यो फिल्म धेरै राम्रो छ
1,मलाई यो सेवा मन पर्यो,मलाई यो सेवा मन पर्यो
2,खाना निकै मिठो थियो,खाना निकै मिठो थियो
3,अनुभव उत्कृष्ट भयो,अनुभव उत्कृष्ट भयो
4,यो उत्पादन उपयोगी छ,यो उत्पादन उपयोगी छ


In [9]:
nepali_stopwords = {
    'अनि', 'अथवा', 'अझै', 'आज', 'त्यसैले', 'त्यसको', 'त्यसमा', 'त्यो', 'त', 'तर',
    'तथा', 'तिमी', 'तिम्रो', 'तिनले', 'ती', 'थिए', 'थियो', 'छ', 'छु', 'छन्', 'छैन',
    'जब', 'जसले', 'जसमा', 'जुन', 'जे', 'जो', 'न', 'नि', 'पनि', 'पर्‍यो', 'पहिले',
    'भए', 'भएको', 'भने', 'भन्ने', 'भित्र', 'म', 'मा', 'मात्र', 'मेरो', 'यति',
    'यदि', 'यस', 'यसका', 'यसको', 'यसले', 'यहाँ', 'या', 'र', 'रही', 'रहेका',
    'लिए', 'लाई', 'ले', 'वा', 'संग', 'सँग', 'सबै', 'हो', 'हुन', 'हुन्छ', 'हुन्'
}

def remove_stopwords(text):
    return ' '.join(word for word in text.split() if word not in nepali_stopwords)

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
df = df[df['clean_text'].ne('')].copy()
df[['text', 'clean_text']].head()

,text,clean_text
0,यो फिल्म धेरै राम्रो छ,यो फिल्म धेरै राम्रो
1,मलाई यो सेवा मन पर्यो,मलाई यो सेवा मन पर्यो
2,खाना निकै मिठो थियो,खाना निकै मिठो
3,अनुभव उत्कृष्ट भयो,अनुभव उत्कृष्ट भयो
4,यो उत्पादन उपयोगी छ,यो उत्पादन उपयोगी


In [10]:
# Save processed data
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f'Saved processed data to {PROCESSED_DATA_PATH}')

Saved processed data to /Users/mrs/ML LAB/ML PRoject/nepali-sentiment-analysis/data/processed/nepali_sentiment_clean.csv


## 4. Train/Test Split & TF-IDF Vectorization

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

label_counts = df['label'].value_counts()
stratify_labels = df['label'] if label_counts.min() >= 2 else None
if stratify_labels is None:
    print('Warning: at least one class has fewer than 2 samples; using an unstratified split.')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=stratify_labels
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    token_pattern=r'(?u)[\u0900-\u097F]+',
)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

X_train.shape, X_test.shape

((48, 211), (12, 211))

## 5. Model Training

In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import clone

models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Linear SVM': LinearSVC(class_weight='balanced')
}

for name, model in models.items():
    model.fit(X_train, y_train)

## 6. Evaluation

In [13]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

results = []
for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average='macro', zero_division=0
    )
    results.append({'model': name, 'accuracy': acc, 'precision': precision, 'recall': recall, 'f1_macro': f1})
    print(f'--- {name} ---')
    print(classification_report(y_test, preds, zero_division=0))

results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
results_df

--- Naive Bayes ---
              precision    recall  f1-score   support

    negative       0.29      0.50      0.36         4
     neutral       0.67      0.50      0.57         4
    positive       0.00      0.00      0.00         4

    accuracy                           0.33        12
   macro avg       0.32      0.33      0.31        12
weighted avg       0.32      0.33      0.31        12

--- Logistic Regression ---
              precision    recall  f1-score   support

    negative       0.25      0.25      0.25         4
     neutral       0.50      0.75      0.60         4
    positive       0.00      0.00      0.00         4

    accuracy                           0.33        12
   macro avg       0.25      0.33      0.28        12
weighted avg       0.25      0.33      0.28        12

--- Linear SVM ---
              precision    recall  f1-score   support

    negative       0.33      0.25      0.29         4
     neutral       0.57      1.00      0.73         4
    posi

,model,accuracy,precision,recall,f1_macro
0,Linear SVM,0.500000,0.468254,0.500000,0.448773
1,Naive Bayes,0.333333,0.317460,0.333333,0.311688
2,Logistic Regression,0.333333,0.250000,0.333333,0.283333


In [14]:
results_path = RESULTS_DIR / 'model_comparison.csv'
results_df.to_csv(results_path, index=False)
print(f'Saved model comparison to {results_path}')

Saved model comparison to /Users/mrs/ML LAB/ML PRoject/nepali-sentiment-analysis/results/model_comparison.csv


In [15]:
# Confusion matrix for the best model
best_model_name = results_df.sort_values('f1_macro', ascending=False).iloc[0]['model']
best_model = models[best_model_name]
preds = best_model.predict(X_test)

labels = list(best_model.classes_)
cm = confusion_matrix(y_test, preds, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## 7. Results Summary & Model Saving

In [16]:
final_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    token_pattern=r'(?u)[\u0900-\u097F]+',
)
X_all = final_vectorizer.fit_transform(df['clean_text'])
final_model = clone(best_model)
final_model.fit(X_all, df['label'])

model_artifact = {
    'model': final_model,
    'vectorizer': final_vectorizer,
    'best_model_name': best_model_name,
    'metrics': results_df.to_dict(orient='records'),
    'stopwords': sorted(nepali_stopwords),
}

model_path = MODELS_DIR / 'final_model.pkl'
joblib.dump(model_artifact, model_path)
print(f'Saved best model ({best_model_name}) and vectorizer to {model_path}')

Saved best model (Linear SVM) and vectorizer to /Users/mrs/ML LAB/ML PRoject/nepali-sentiment-analysis/models/final_model.pkl


In [17]:
def predict_sentiment(text):
    cleaned = remove_stopwords(clean_text(text))
    features = final_vectorizer.transform([cleaned])
    return final_model.predict(features)[0]

# Example usage after training:
# predict_sentiment('यो फिल्म निकै राम्रो छ')

### Summary

- Best performing model: selected automatically by highest macro F1 score in `results_df`.
- Key takeaways: TF-IDF features with linear models provide a strong, interpretable baseline for Nepali sentiment classification.
- Limitations: performance depends on dataset quality, class balance, spelling variation, informal Nepali, and mixed Devanagari/Romanized text.